[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jman4162/sensortwin-transformer-agent/blob/master/notebooks/03_transformer_training_colab.ipynb)

# SensorTwin — train `SensorPatchTST` on a Colab GPU

A self-contained Colab walkthrough: install the package, generate the `colab_standard` synthetic
benchmark (20k samples), and train the patch-transformer on a GPU. Mixed precision and a pinned
multi-worker DataLoader turn on automatically when CUDA is present (CPU runs stay FP32 and
bit-identical), so no extra flags are needed.

Runtime: **Runtime → Change runtime type → GPU** before running. Heavier sweeps
(label-efficiency, robustness, the agentic runner) are one `make` target each — see the last cell.

In [ ]:
# Opened straight from the Colab badge? Only this notebook is present, so clone the repo first.
# The repo is private: clone with a GitHub token that has access, e.g.
#   !git clone https://<TOKEN>@github.com/jman4162/sensortwin-transformer-agent.git
# (or make the repo public for a frictionless one-click open).
import os

if not os.path.exists("sensortwin"):
    !git clone https://github.com/jman4162/sensortwin-transformer-agent.git
    %cd sensortwin-transformer-agent
%pip install -q -e ".[ml]"

In [ ]:
import torch

print("torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected — this still runs on CPU (slower; AMP stays off).")

In [ ]:
import numpy as np

from sensortwin.data.dataset import SensorArrayDataset
from sensortwin.data.splits import make_split
from sensortwin.data.transforms import ChannelStandardizer
from sensortwin.simulation import GenConfig, generate_dataset
from sensortwin.simulation.events import EVENT_CLASSES
from sensortwin.utils.config import load_yaml
from sensortwin.utils.seeds import set_torch_seed

set_torch_seed(0)

# colab_standard = 20k samples. Drop to 4000 if you want a faster first pass.
X, y, _ = generate_dataset(GenConfig(n_samples=20_000, T=512, seed=0, normalize=False))
splits = make_split("random", y, {}, seed=0)
std = ChannelStandardizer().fit(X[splits["train"]])
print("X", X.shape, "classes", len(EVENT_CLASSES))

In [ ]:
from sensortwin.models.transformer import SensorPatchTST
from sensortwin.training.augment import build_augment
from sensortwin.training.loop import class_weights, predict_proba, train_model

cfg = load_yaml("configs/models/sensorpatchtst.yaml")
model = SensorPatchTST(**cfg["model"])
tcfg = cfg["train"]
loop_keys = ("optimizer", "lr", "weight_decay", "label_smoothing", "scheduler",
             "warmup_epochs", "batch_size", "patience")
kw = {k: tcfg[k] for k in loop_keys if k in tcfg}
aug = build_augment(tcfg.get("augment"))
if aug is not None:
    kw["augment"] = aug

tr, va, te = splits["train"], splits["val"], splits["test"]
train_ds = SensorArrayDataset(std.transform(X[tr]), y[tr]).as_torch()
val_ds = SensorArrayDataset(std.transform(X[va]), y[va]).as_torch()
test_ds = SensorArrayDataset(std.transform(X[te]), y[te]).as_torch()

# device=None auto-detects CUDA; AMP + pinned workers switch on there automatically.
model, history = train_model(
    model, train_ds, val_ds, epochs=15,
    weight=class_weights(y[tr], len(EVENT_CLASSES)), **kw,
)
print("best val macro-F1:", round(history.best_val_macro_f1, 3))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history.train_loss, label="train")
ax[0].plot(history.val_loss, label="val")
ax[0].set_title("loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(history.val_macro_f1)
ax[1].set_title("val macro-F1"); ax[1].set_xlabel("epoch")
plt.tight_layout(); plt.show()

In [ ]:
from sensortwin.evaluation.metrics import classification_metrics
from sensortwin.evaluation.plots import plot_confusion_matrix

proba = predict_proba(model, test_ds)
m = classification_metrics(y[te], proba.argmax(1), proba, EVENT_CLASSES)
print("test macro-F1:", round(m["macro_f1"], 3), "| macro-AUROC:", round(m["macro_auroc"], 3))
plot_confusion_matrix(np.array(m["confusion_matrix"]), EVENT_CLASSES, "cm.png")
from IPython.display import Image
Image("cm.png")

## Heavier studies (each is one command)

All auto-detect the GPU. Run at `--mode colab_standard` for research-grade numbers.

```bash
!python -m scripts.train_baseline --models logreg,xgboost,cnn,lstm,transformer --mode colab_standard --epochs 15
!python -m scripts.label_efficiency_sweep --mode colab_standard --epochs-pretrain 50 --epochs-finetune 30 --seeds 3
!python -m scripts.robustness_report --mode colab_standard --epochs 20
!python -m scripts.run_agent --epochs 12 --seeds 5            # agentic ablation campaign
```

These are wiring-grade at `quick_demo`; the synthetic numbers are a controlled testbed, not real-world
validation (see `reports/model_card.md`).